# 🎯 Stage Gate G1 — correlation pre-test

**Why this gate exists.** Before you spend GPU-hours on RL with a mechanistic reward, you need evidence that your SAE feature pack (helpful features minus harmful features) actually *correlates* with outcome correctness. If the correlation is weak, RL will either not move the needle or, worse, optimize a proxy that diverges from the real objective.

**Protocol.** G1 → G2 → G3 (see [OpenInterpretability/mechreward](https://github.com/OpenInterpretability/mechreward)).

**Pass threshold.** Spearman `ρ ≥ 0.30` (with `p < 0.05`) on a held-out slice of the evaluation dataset. Anything below means: **refine the pack before burning GPU**.

**Runtime.** 20–30 min on a T4 for 100 GSM8K samples × 256 new tokens on a 2–4B model. Scale up GPU for bigger bases.

**Deliverable.** `g1_report.json` with per-example `R_mech` + correctness, uploaded to the SAE repo.

In [ ]:
!pip install -q transformers==4.57.1 accelerate==1.12.0 datasets==4.0.0 safetensors==0.4.5 huggingface_hub==1.5.0 scipy tqdm matplotlib

## Config

Edit the `PACK` dict to match the feature IDs in your SAE that you believe are **helpful** (rise when reasoning is going well) vs **harmful** (rise when the model is bluffing/confabulating). Feature IDs come from `feature_catalog.json` produced by notebook `04_discover_features.ipynb`, or from your own inspection.

In [ ]:
HF_SAE_REPO       = 'YOUR_USER/your-sae'
HF_BASE_MODEL     = 'google/gemma-2-2b'
LAYER             = 15
D_MODEL           = 2304
D_SAE             = 16384
K                 = 64

# Feature pack — replace with the IDs you picked from feature_catalog.json.
PACK = {
    'helpful': [1234, 5678, 9012],   # e.g. 'step_by_step_arith', 'unit_cancellation', 'checks_work'
    'harmful': [3456, 7890, 1357],   # e.g. 'confident_assertion_without_math', 'bluff_phrase', 'answer_guess'
}

EVAL_DATASET      = 'openai/gsm8k'
EVAL_SUBSET       = 'main'
EVAL_SPLIT        = 'test'
N_SAMPLES         = 100
MAX_NEW_TOKENS    = 256
TEMPERATURE       = 0.0               # greedy — we want reproducible per-example R_mech
SEED              = 0

PASS_THRESHOLD    = 0.30

import os, json, math, random, time
random.seed(SEED)
print('Config loaded.')
print(f'  SAE      : {HF_SAE_REPO}')
print(f'  Base     : {HF_BASE_MODEL}  layer={LAYER}')
print(f'  Pack     : {len(PACK["helpful"])} helpful, {len(PACK["harmful"])} harmful')
print(f'  Eval     : {EVAL_DATASET}/{EVAL_SUBSET} split={EVAL_SPLIT}  N={N_SAMPLES}')
print(f'  Threshold: ρ ≥ {PASS_THRESHOLD}')

## Auth

Set `HF_TOKEN` — required to download the SAE and to upload `g1_report.json` at the end. On Colab use the 🔑 Secrets panel; outside Colab, set the env var.

In [ ]:
def _get_secret(name):
    v = os.environ.get(name)
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        try:
            return userdata.get(name)
        except Exception:
            return None
    except Exception:
        return None

HF_TOKEN = _get_secret('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(HF_TOKEN, add_to_git_credential=False)
    print('HF logged in.')
else:
    print('WARNING: no HF_TOKEN — SAE download and final upload will fail.')

## Load SAE + base model (frozen)

TopK SAE with alias-reconciliation for the common formats (`W_enc/W_dec/b_enc/b_dec` and `encoder.weight/decoder.weight/...`). Base model loaded in `bfloat16` with SDPA. All parameters frozen — we're not training anything, just reading activations.

In [ ]:
import torch
import torch.nn as nn
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, list_repo_files
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# ---- SAE ----
_candidate_files = [
    'sae_final.safetensors',
    f'sae_L{LAYER}_latest.safetensors',
    f'sae_L{LAYER}.safetensors',
    'sae.safetensors',
]
repo_files = list_repo_files(HF_SAE_REPO)
sae_file = next((f for f in _candidate_files if f in repo_files), None)
if sae_file is None:
    sae_file = next((f for f in repo_files if f.endswith('.safetensors') and 'sae' in f.lower()), None)
assert sae_file is not None, f'No SAE .safetensors found in {HF_SAE_REPO}. Files: {repo_files}'
print('Using SAE file:', sae_file)

sae_path = hf_hub_download(HF_SAE_REPO, sae_file)
sd = load_file(sae_path)

class TopKSAE(nn.Module):
    def __init__(self, d_model, d_sae, k):
        super().__init__()
        self.d_model, self.d_sae, self.k = d_model, d_sae, k
        self.W_enc = nn.Parameter(torch.zeros(d_model, d_sae))
        self.W_dec = nn.Parameter(torch.zeros(d_sae, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    @torch.no_grad()
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = torch.topk(pre, self.k, dim=-1)
        vals = torch.relu(vals)
        out = torch.zeros_like(pre)
        out.scatter_(-1, idx, vals)
        return out

sae = TopKSAE(D_MODEL, D_SAE, K)

def _assign(param_name, tensor):
    p = getattr(sae, param_name)
    if tensor.shape != p.shape:
        if tensor.T.shape == p.shape:
            tensor = tensor.T
        else:
            raise ValueError(f'shape mismatch for {param_name}: {tensor.shape} vs {p.shape}')
    p.data.copy_(tensor.to(p.dtype))

aliases = {
    'W_enc': ['W_enc', 'encoder.weight', 'encoder.W', 'enc.weight'],
    'W_dec': ['W_dec', 'decoder.weight', 'decoder.W', 'dec.weight'],
    'b_enc': ['b_enc', 'encoder.bias',   'enc.bias'],
    'b_dec': ['b_dec', 'decoder.bias',   'dec.bias', 'pre_bias'],
}
for tgt, names in aliases.items():
    for n in names:
        if n in sd:
            _assign(tgt, sd[n]); break
    else:
        print(f'WARNING: no source for {tgt}')

sae = sae.to(device=device, dtype=torch.bfloat16).eval()
for p in sae.parameters():
    p.requires_grad_(False)

# Sanity — the feature IDs in PACK must exist in [0, D_SAE)
for role, ids in PACK.items():
    for fid in ids:
        assert 0 <= fid < D_SAE, f'{role} feature {fid} out of range [0, {D_SAE})'
print('SAE ready. Pack IDs in range.')

# ---- Base model ----
tokenizer = AutoTokenizer.from_pretrained(HF_BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map={'': device},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# Locate decoder layer list (covers dense / multimodal / hybrid).
_candidates = [
    getattr(getattr(model, 'model', None), 'layers', None),
    getattr(getattr(getattr(model, 'model', None), 'language_model', None), 'layers', None),
    getattr(getattr(model, 'language_model', None), 'layers', None),
    getattr(getattr(getattr(model, 'model', None), 'decoder', None), 'layers', None),
    getattr(getattr(model, 'transformer', None), 'h', None),
]
layers = next((c for c in _candidates if c is not None), None)
assert layers is not None, 'Could not locate decoder layers on this model.'
print(f'Hooking layer {LAYER} of {len(layers)} ({type(layers[LAYER]).__name__})')

_captured = {}
def _hook(module, inputs, output):
    h = output[0] if isinstance(output, tuple) else output
    _captured['h'] = h.detach()

hook_handle = layers[LAYER].register_forward_hook(_hook)
print('Hook registered.')

## Generate + score

For each of `N_SAMPLES` GSM8K test examples we:

1. Greedy-generate up to `MAX_NEW_TOKENS` tokens (temperature=0 so `R_mech` is deterministic).
2. Re-run a forward pass over the full sequence to capture residuals at `LAYER`.
3. Encode through the SAE and average **only the response-token** activations of each pack feature.
4. Compute `R_mech = mean(helpful_feat_activations) − mean(harmful_feat_activations)`.
5. Parse the numeric answer and compare to the gold `#### N` label.

We capture-then-score (rather than stream during generation) to keep the hook simple and to avoid KV-cache state issues across architectures.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

ds = load_dataset(EVAL_DATASET, EVAL_SUBSET, split=EVAL_SPLIT)
# Deterministic random sample (don't shuffle all of test).
rng = random.Random(SEED)
indices = rng.sample(range(len(ds)), min(N_SAMPLES, len(ds)))
samples = ds.select(indices)
print(f'Selected {len(samples)} GSM8K examples (seed={SEED}).')

helpful_ids = torch.tensor(PACK['helpful'], dtype=torch.long)
harmful_ids = torch.tensor(PACK['harmful'], dtype=torch.long)

# GSM8K answer parsing.
def gold_answer(example):
    tail = example['answer'].split('####')[-1].strip()
    tail = tail.replace(',', '').split()[0]
    return float(tail)

def pred_answer(text):
    # prefer '#### N' if the model copied that format; fallback to the last number in the response
    if '####' in text:
        tail = text.split('####')[-1].strip()
        tok = tail.replace(',', '').split()
        if tok:
            try:
                return float(tok[0])
            except ValueError:
                pass
    # fallback: last numeric token in the whole response
    import re
    nums = re.findall(r'-?\d+(?:\.\d+)?', text.replace(',', ''))
    if nums:
        try:
            return float(nums[-1])
        except ValueError:
            return None
    return None

PROMPT_TEMPLATE = (
    'Solve the problem step by step. End with "#### <number>".\n\n'
    'Problem: {q}\n\n'
    'Solution:'
)

R_mech = []
correctness = []
per_example = []

for ex in tqdm(samples, desc='G1 eval'):
    question = ex['question']
    gold = gold_answer(ex)
    prompt = PROMPT_TEMPLATE.format(q=question)

    enc = tokenizer(prompt, return_tensors='pt').to(device)
    prompt_len = int(enc['input_ids'].shape[1])

    # 1) generate
    with torch.no_grad():
        gen = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None if TEMPERATURE == 0.0 else TEMPERATURE,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )
    full_ids = gen[0]                                             # [T_total]
    response_ids = full_ids[prompt_len:]                          # [T_resp]
    response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
    resp_len = int(response_ids.shape[0])

    if resp_len == 0:
        # Model produced nothing — skip but record.
        per_example.append({
            'question': question, 'gold': gold, 'pred': None,
            'response_text': response_text, 'resp_len': 0,
            'R_mech': 0.0, 'correct': 0,
            'helpful_mean': 0.0, 'harmful_mean': 0.0,
        })
        R_mech.append(0.0); correctness.append(0)
        continue

    # 2) capture residuals via a single forward over the full sequence
    with torch.no_grad():
        _ = model(input_ids=full_ids.unsqueeze(0), use_cache=False)
    h = _captured['h']                                            # [1, T_total, D_MODEL]
    # Response tokens sit at [prompt_len : prompt_len + resp_len)
    h_resp = h[:, prompt_len:prompt_len + resp_len, :]            # [1, T_resp, D_MODEL]

    # 3) SAE encode + per-feature mean over response tokens
    z = sae.encode(h_resp.to(torch.bfloat16))                     # [1, T_resp, D_SAE]
    z = z.float().squeeze(0)                                      # [T_resp, D_SAE]
    helpful_act = z[:, helpful_ids].mean(dim=0)                   # [n_helpful]
    harmful_act = z[:, harmful_ids].mean(dim=0)                   # [n_harmful]
    helpful_mean = float(helpful_act.mean().item())
    harmful_mean = float(harmful_act.mean().item())
    r_mech = helpful_mean - harmful_mean

    # 4) grade
    pred = pred_answer(response_text)
    correct = int(pred is not None and abs(pred - gold) < 1e-4)

    R_mech.append(r_mech)
    correctness.append(correct)
    per_example.append({
        'question': question, 'gold': gold, 'pred': pred,
        'response_text': response_text, 'resp_len': resp_len,
        'R_mech': r_mech, 'correct': correct,
        'helpful_mean': helpful_mean, 'harmful_mean': harmful_mean,
        'per_helpful': helpful_act.tolist(),
        'per_harmful': harmful_act.tolist(),
    })

hook_handle.remove()
acc = sum(correctness) / max(len(correctness), 1)
print(f'Done. accuracy = {acc:.3f} ({sum(correctness)}/{len(correctness)})')
print(f'R_mech stats: mean={sum(R_mech)/len(R_mech):+.4f}  min={min(R_mech):+.4f}  max={max(R_mech):+.4f}')

## The gate

Spearman ρ is the standard choice here — it only cares about rank order, so it's robust to the (often heavy-tailed) activation scale of SAE features. We still report Pearson as a sanity check.

| Decision | ρ               | Next step                                                    |
|----------|-----------------|--------------------------------------------------------------|
| ✅ Pass  | ≥ 0.30 (p<0.05) | Proceed to **G2** — reward-model sanity & sign-flip checks.  |
| ❌ Fail  | < 0.30          | **Do not train.** Refine the pack (swap features, add more, re-inspect snippets), or pick a different `LAYER`. |

In [ ]:
import numpy as np

R = np.asarray(R_mech, dtype=float)
C = np.asarray(correctness, dtype=float)

# Prefer scipy Spearman; fall back to numpy Pearson-on-ranks if scipy unavailable.
try:
    from scipy.stats import spearmanr, pearsonr
    rho, p_rho = spearmanr(R, C)
    r_pearson, p_pearson = pearsonr(R, C)
    used_scipy = True
except Exception:
    used_scipy = False
    def _rank(a):
        order = np.argsort(a, kind='mergesort')
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(len(a))
        return ranks
    rho = float(np.corrcoef(_rank(R), _rank(C))[0, 1])
    r_pearson = float(np.corrcoef(R, C)[0, 1])
    # approximate p-value for Spearman via t-distribution
    n = len(R)
    if n > 2 and abs(rho) < 1.0:
        t = rho * math.sqrt((n - 2) / max(1e-12, 1 - rho * rho))
        # two-sided normal approximation (crude but fine for reporting)
        from math import erf, sqrt
        p_rho = 2 * (1 - 0.5 * (1 + erf(abs(t) / sqrt(2))))
    else:
        p_rho = float('nan')
    p_pearson = float('nan')

print(f'Spearman ρ = {rho:+.3f}   p = {p_rho:.1e}   (scipy={used_scipy})')
print(f'Pearson  r = {r_pearson:+.3f}   p = {p_pearson:.1e}')

if rho >= PASS_THRESHOLD and (p_rho < 0.05 if not math.isnan(p_rho) else True):
    verdict = 'PASS'
    print(f'\n✅ G1 PASSED — proceed to G2.  (ρ={rho:+.3f} ≥ {PASS_THRESHOLD})')
else:
    verdict = 'FAIL'
    print(f'\n❌ G1 FAILED — refine pack before burning GPU.  (ρ={rho:+.3f} < {PASS_THRESHOLD})')
    print('   Suggestions:')
    print('    • re-inspect top-activating snippets of each pack feature (notebook 04)')
    print('    • try a different LAYER (mid-to-late residual usually wins for reasoning)')
    print('    • enlarge the pack: 5–10 features per side is healthier than 3')
    print('    • check for sign flips — swap a feature between helpful/harmful')

# ---- Save + upload report ----
report = {
    'sae_repo': HF_SAE_REPO,
    'base_model': HF_BASE_MODEL,
    'layer': LAYER,
    'd_model': D_MODEL,
    'd_sae': D_SAE,
    'k': K,
    'eval_dataset': EVAL_DATASET,
    'eval_subset': EVAL_SUBSET,
    'eval_split': EVAL_SPLIT,
    'n_samples': len(R),
    'max_new_tokens': MAX_NEW_TOKENS,
    'temperature': TEMPERATURE,
    'seed': SEED,
    'pack': PACK,
    'pass_threshold': PASS_THRESHOLD,
    'accuracy': float(C.mean()),
    'spearman_rho': float(rho),
    'spearman_p': float(p_rho) if not math.isnan(p_rho) else None,
    'pearson_r': float(r_pearson),
    'pearson_p': float(p_pearson) if not math.isnan(p_pearson) else None,
    'verdict': verdict,
    'per_example': per_example,
}
with open('g1_report.json', 'w') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f'\nWrote g1_report.json ({os.path.getsize("g1_report.json")} bytes)')

if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj='g1_report.json',
        path_in_repo='g1_report.json',
        repo_id=HF_SAE_REPO,
        repo_type='model',
        commit_message=f'Stage Gate G1: ρ={rho:+.3f} verdict={verdict} N={len(R)}',
    )
    print(f'Uploaded to https://huggingface.co/{HF_SAE_REPO}/blob/main/g1_report.json')
else:
    print('Skipping upload — no HF_TOKEN.')

## Visualize

Scatter `R_mech` (x) vs correctness (y, jittered). A visible upward trend in the right-hand column of points (correct=1) vs the left-hand column (correct=0) confirms the ρ number is not a fluke of a few outliers.

In [ ]:
import matplotlib.pyplot as plt

rng_plot = np.random.default_rng(SEED)
jitter = rng_plot.uniform(-0.04, 0.04, size=len(C))
C_jit = C + jitter

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d9534f' if c == 0 else '#2b8a3e' for c in correctness]
ax.scatter(R, C_jit, c=colors, alpha=0.75, s=42, edgecolor='white', linewidth=0.6)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
ax.axvline(0.0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('R_mech  =  mean(helpful feat act)  −  mean(harmful feat act)')
ax.set_ylabel('correct  (0 / 1, jittered)')
ax.set_yticks([0, 1])
ax.set_ylim(-0.25, 1.25)
title_colour = '#2b8a3e' if verdict == 'PASS' else '#d9534f'
ax.set_title(f'Stage Gate G1 — ρ = {rho:+.3f}   (N={len(R)}, acc={C.mean():.2f})   [{verdict}]',
             color=title_colour, fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('g1_scatter.png', dpi=140)
plt.show()
print('Saved g1_scatter.png')

# Optional: upload the plot too, so the repo reviewer sees it at a glance.
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj='g1_scatter.png',
        path_in_repo='g1_scatter.png',
        repo_id=HF_SAE_REPO,
        repo_type='model',
        commit_message=f'Stage Gate G1 scatter (ρ={rho:+.3f}, {verdict})',
    )
    print('Uploaded g1_scatter.png')